# Odor Classifier Training
5-class multi-label odor prediction: fruity, sweet, sulfurous, woody, fatty

In [ ]:
%pip install scikit-learn

In [ ]:
!git clone https://github.com/FufanLu/molecular-foundation-model.git

### Upload embeddings

In [ ]:
import os
os.makedirs('molecular-foundation-model/data/processed/embeddings', exist_ok=True)
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    !mv {name} molecular-foundation-model/data/processed/embeddings/

### Load data + encode labels

In [ ]:
import sys
sys.path.append('molecular-foundation-model')

from src.evaluation.similarity import load_embeddings
from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

embeddings = load_embeddings()
df = load_leffingwell()
df = clean_smiles(df)
df = encode_labels(df)

print(f"{len(embeddings)} embeddings, {len(df)} molecules")
label_distribution(df)

### Train Baseline MLP (on frozen embeddings)

In [ ]:
from src.classifier.train import split_data, train_baseline, evaluate_baseline

compounds, X, Y, train_idx, test_idx = split_data(embeddings, df, test_size=0.2, seed=42)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

model, best_f1 = train_baseline(
    X[train_idx], Y[train_idx],
    X[test_idx], Y[test_idx],
    epochs=100, lr=1e-3, patience=15
)

In [ ]:
evaluate_baseline(model, X[test_idx], Y[test_idx], ALL_5_CLASSES)

### Save baseline model

In [ ]:
import torch
torch.save(model.state_dict(), 'odor_baseline.pt')
from google.colab import files
files.download('odor_baseline.pt')

## LoRA Fine-tuning (optional, needs GPU)

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
%pip install peft

In [ ]:
import unimol_hf
from transformers import AutoModel, AutoTokenizer
from importlib.resources import files

pretrained = files('unimol_hf').joinpath('pretrained/unimol-v1-noh')
tokenizer = AutoTokenizer.from_pretrained(str(pretrained))
model = AutoModel.from_pretrained(str(pretrained))

print("Model structure:")
for name, _ in model.named_modules():
    if 'q_proj' in name or 'k_proj' in name or 'v_proj' in name or 'attn' in name:
        print(f"  {name}")
    if 'Linear' in str(type(_)) and ('query' in name.lower() or 'key' in name.lower() or 'value' in name.lower()):
        print(f"  ATTENTION: {name}")

In [ ]:
from src.classifier.model import apply_lora, OdorClassifier

n = apply_lora(model, r=4, alpha=8, target_names=['q_proj', 'k_proj', 'v_proj'])
print(f"LoRA applied to {n} layers")

classifier = OdorClassifier(model, hidden_dim=128, num_classes=5, dropout=0.2)
classifier = classifier.cuda() if torch.cuda.is_available() else classifier

trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from unimol_hf import UnimolDataCollator
import numpy as np

collator = UnimolDataCollator(pad_token_id=tokenizer.pad_token_id)

train_smiles = [df.iloc[i]['smiles'] for i in train_idx]
test_smiles = [df.iloc[i]['smiles'] for i in test_idx]

print(f"Tokenizing {len(train_smiles)} train + {len(test_smiles)} test SMILES...")
train_tokens = [tokenizer.encode(s) for s in train_smiles]
test_tokens = [tokenizer.encode(s) for s in test_smiles]

print("Done.")

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
device = next(classifier.parameters()).device

epochs = 30
batch_size = 16
train_y = torch.tensor(Y[train_idx], dtype=torch.float32).to(device)
test_y = torch.tensor(Y[test_idx], dtype=torch.float32).to(device)

for epoch in range(epochs):
    classifier.train()
    perm = torch.randperm(len(train_tokens))
    total_loss = 0.0

    for i in range(0, len(train_tokens), batch_size):
        idx = perm[i:i+batch_size]
        batch_tokens = [train_tokens[j] for j in idx]
        batch_y = train_y[idx]

        batch = collator(batch_tokens)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = classifier(batch)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0 or epoch == epochs - 1:
        classifier.eval()
        with torch.no_grad():
            test_batch = collator(test_tokens)
            test_batch = {k: v.to(device) for k, v in test_batch.items()}
            test_preds = torch.sigmoid(classifier(test_batch)).cpu().numpy()
            test_bin = (test_preds > 0.5).astype(int)
            f1 = f1_score(Y[test_idx], test_bin, average='macro', zero_division=0)
        print(f"epoch {epoch:3d}  loss {total_loss/max(1,(len(train_tokens)//batch_size)):.4f}  f1_macro {f1:.4f}")
        classifier.train()

In [ ]:
torch.save(classifier.state_dict(), 'odor_lora.pt')
files.download('odor_lora.pt')